# Search Engine Evaluation

An evaluation showcase of the search engine implemented by the `SearchEngine`
class using sparse retrieval (BM25) on HotPotQA dataset.


In [1]:
from typing import cast

import pandas as pd
from datasets import load_dataset

from reactagenthlt.tools import SearchEngine

## General Setting

The following code will evaluate the search engine just by using the `question`
field of HotPotQA as a query. The search engine will return only the top 2
results whose title will be compared to the 2 **supporting facts** titles for
the question.

The final metric will be just the overall **accuracy** that of course cannot be
very high due to the _multi-hop_ reasoning required to answer.


In [2]:
ds = cast(
    pd.DataFrame,
    load_dataset("hotpotqa/hotpot_qa", "distractor")["validation"].to_pandas(),
)
ds = ds[["question", "supporting_facts"]][:1000]
ds["supporting_facts"] = ds["supporting_facts"].apply(lambda x: x["title"])
ds

,question,supporting_facts
0,Were Scott Derrickson and Ed Wood of the same ...,"[Scott Derrickson, Ed Wood]"
1,What government position was held by the woman...,"[Kiss and Tell (1945 film), Shirley Temple, Sh..."
2,"What science fantasy young adult series, told ...","[The Hork-Bajir Chronicles, The Hork-Bajir Chr..."
3,Are the Laleli Mosque and Esma Sultan Mansion ...,"[Laleli Mosque, Esma Sultan Mansion]"
4,"The director of the romantic comedy ""Big Stone...","[Big Stone Gap (film), Adriana Trigiani]"
...,...,...
995,Who is the author of the play that was adapted...,"[Suite from Henry V, Henry V (1944 film)]"
996,Which type of character is featured by the P.L...,"[Mary Poppins Opens the Door, Mary Poppins (ch..."
997,What distinction about his birth makes the con...,"[Bill Pollack, Bill Pollack, Phil Hill]"
998,Are both Yangzhou and Jiangyan District consid...,"[Yangzhou, Yangzhou, Jiangyan District]"


In [3]:
se = SearchEngine(search_by="document", top_k=2)

building corpus: 100%|███████████████████| 7405/7405 [00:00<00:00, 25111.04it/s]


In [4]:
questions = ds["question"].to_list()
titles = ds["supporting_facts"].to_list()

correct = 0.0
total = 0.0
for q, t in zip(questions, titles):
    res = se.search(q)["title"].to_list()
    for r in res:
        correct += 1.0 if r in t else 0.0
        total += 1.0
        print(
            f"done: {int(total)}/{2 * len(questions)}, correct: {int(correct)}, accuracy: {correct / total:.2f}",
            end="\r",
        )


## Results

The fact that the search engine obtains low accuracy close to $0.54$ is
perfectly coherent with the _multi-hop_ nature of the dataset.

Using the question as query basically targets the first document that should be
retrieved in the trajectory of the agent.

To correctly retrieve also the second document there is need for a query that is
the result of the observation obtained from the first document, the question and
the _self-reasoning_ of the agent.
